## Notebook content

This notebook is for **AutoGluon Time Series** models produced by the timeseries training pipeline. It helps you:

- Load a refitted model artifact from S3 (same run as the pipeline).
- Inspect test metrics and predictor metadata.
- Run **`TimeSeriesPredictor.predict()`** to forecast the next `prediction_length` steps per time series.

**Tips**

- Configure S3 access (workbench secret or `.env`) so the notebook can download `model_artifact/.../<MODEL>_FULL/`.
- `model_name` must match the refitted model folder (e.g. `ETS_FULL`, `Naive_FULL`), as in the pipeline leaderboard.
- For `predict`, pass historical rows per `item_id` (target column + timestamps). The model forecasts after the last timestamp you provide—not a single “score row” like tabular classification.

### Contents

**[Setup](#setup)**  
**[Experiment run details](#experiment-run-details)**  
**[Download trained model](#download-trained-model)**  
**[Model insights](#model-insights)**  
**[Back-testing metrics](#back-testing-metrics)**  
**[Back-testing charts](#back-testing-charts)**  
**[Load the predictor](#load-the-predictor)**  
**[Refit training leaderboard](#refit-training-leaderboard)**  
**[Forecast (predict)](#predict-the-values)**  
**[Plot forecasted data](#plot-forecasted-data)**  
**[Summary and next steps](#summary-and-next-steps)**

<a id="setup"></a>
## Setup

In [ ]:
import warnings

warnings.filterwarnings("ignore")

In [ ]:
import os

os.environ["PIP_EXTRA_INDEX_URL"] = (
    "https://console.redhat.com/api/pypi/public-rhai/rhoai/3.5/cpu-ubi9-test/simple/"
)

%pip install autogluon.timeseries==1.5.0+rhaiv.4 | tail -n 1


<a id="experiment-run-details"></a>
## Experiment run details

Set the pipeline name and run ID that identify the training run whose artifacts you want to load. These values are typically available from the pipeline run or workbench.

In [ ]:
pipeline_name = "autogluon-timeseries-training-pipeline"
run_id = "ba44c73f-307c-4529-84e9-27a0698ae2ae"
model_name = "RecursiveTabular_FULL"

<a id="download-trained-model"></a>
## Download trained model

 📌 **Action:** Ensure the S3 connection is added to the workbench so the notebook can access the results.

Download the refitted model directory from S3: `predictor/`, `metrics/`, `notebooks/` under `models_artifact/<model_name>/`.

The download cell lists objects under `<pipeline_name>/<run_id>/autogluon-timeseries-models-training/` and matches keys containing `models_artifact/<model_name>/`.

In [ ]:
import boto3
import os
import warnings

from botocore.exceptions import SSLError

required_env_vars = (
    "AWS_S3_ENDPOINT",
    "AWS_ACCESS_KEY_ID",
    "AWS_SECRET_ACCESS_KEY",
    "AWS_S3_BUCKET",
)
missing_vars = [name for name in required_env_vars if not os.environ.get(name, "").strip()]

if missing_vars:
    raise ValueError(
        f"Missing required environment variable(s): {', '.join(missing_vars)}. "
        "Attach your S3 connection to this notebook workbench and try again."
    )


def _get_s3_bucket(verify=True):
    return boto3.resource(
        "s3",
        endpoint_url=os.environ["AWS_S3_ENDPOINT"],
        verify=verify,
    ).Bucket(os.environ["AWS_S3_BUCKET"])


def _download_model(bucket):
    full_refit_prefix = os.path.join(pipeline_name, run_id, "autogluon-timeseries-models-training")
    best_model_subpath = os.path.join("models_artifact", model_name)
    marker = best_model_subpath + "/"
    base_dir = os.path.abspath(model_name)
    for obj in bucket.objects.filter(Prefix=full_refit_prefix):
        if obj.key.endswith("/") or marker not in obj.key:
            continue
        relative = obj.key.split(marker, 1)[1]
        relative = os.path.normpath(relative)
        if relative == "." or relative.startswith("..") or os.path.isabs(relative):
            warnings.warn(f"Skipping due to invalid path detected: {obj.key!r}")
            continue
        target = os.path.join(model_name, relative)
        abs_target = os.path.abspath(target)
        if abs_target != base_dir and not abs_target.startswith(base_dir + os.sep):
            warnings.warn(f"Skipping due to invalid path detected: {obj.key!r}")
            continue
        parent = os.path.dirname(abs_target)
        if parent:
            os.makedirs(parent, exist_ok=True)
        bucket.download_file(obj.key, target)
    return model_name


try:
    best_model_path = _download_model(_get_s3_bucket())
except SSLError:
    warnings.warn("SSL error when accessing S3, retrying with verify=False")
    best_model_path = _download_model(_get_s3_bucket(verify=False))

print("Model artifact stored under", best_model_path)

<a id="model-insights"></a>
## Model insights

### Metrics

Metrics computed on the holdout test `TimeSeriesDataFrame` during pipeline refit (e.g. mean_absolute_scaled_error, weighted_quantile_loss). Signs may be flipped so higher is better, per AutoGluon convention.

In [ ]:
import json
import os

import pandas as pd

with open(os.path.join(best_model_path, "metrics", "metrics.json")) as f:
    metrics = pd.json_normalize(json.load(f))

metrics

### Back-testing metrics

Rolling validation scores from ``metrics/back_testing.json`` (AutoGluon-style ``Cutoff …: mean_absolute_scaled_error = …`` lines, per-window table, and overall mean). Scores are positive error values; AutoGluon ``evaluate()`` negates them interactively.

In [ ]:
"""Matplotlib charts for ``metrics/back_testing.json`` (same library as tabular ROC/PR curves)."""

from __future__ import annotations

import math
from pathlib import Path
from typing import Any

import pandas as pd

_ACCENT = "#EE0000"
_NEUTRAL = "#161616"
_CUTOFF = "gray"


def _matplotlib():
    """Import matplotlib on demand (same pattern as ROC/PR cells in tabular notebooks)."""
    try:
        import matplotlib.dates as mdates
        import matplotlib.pyplot as plt
    except ImportError as exc:
        raise ImportError(
            "Matplotlib is required for back-testing charts. Install it with: pip install matplotlib"
        ) from exc

    return plt, mdates


def _metric_display_name(metric: str) -> str:
    """Return a short label for plot titles (abbreviates snake_case to initials)."""
    if "_" not in metric:
        return metric
    return "".join(w[0] for w in metric.split("_") if w).upper()


def _normalize_metric(metric: str) -> str:
    return (metric or "").strip().lstrip("-").upper()


def pick_window_metric(metrics: dict[str, Any], eval_metric: str) -> float | None:
    """Return a finite window metric value, preferring ``eval_metric``."""
    if not metrics:
        return None
    wanted = _normalize_metric(eval_metric)
    for key, value in metrics.items():
        if _normalize_metric(key) == wanted and isinstance(value, (int, float)):
            number = float(value)
            if math.isfinite(number):
                return number
    for value in metrics.values():
        if isinstance(value, (int, float)) and math.isfinite(float(value)):
            return float(value)
    return None


def _resolve_item_id(item_id: Any, available_item_ids: list[Any]) -> Any | None:
    """Map a backtest ``item_id`` to the matching ID in ``available_item_ids``."""
    if item_id is None:
        return None
    if item_id in available_item_ids:
        return item_id
    item_text = str(item_id)
    for candidate in available_item_ids:
        if str(candidate) == item_text:
            return candidate
    return None


def backtest_highlight_item_ids(
    back_testing: dict[str, Any],
    available_item_ids: list[Any],
    *,
    max_items: int = 4,
) -> list[Any]:
    """Return best/worst performer IDs from backtest JSON that exist in the sample data."""
    if not available_item_ids or max_items <= 0:
        return []

    series_analysis = back_testing.get("series_analysis") or {}
    resolved: list[Any] = []
    for role in ("best_performer", "worst_performer"):
        performer = series_analysis.get(role)
        if not performer:
            continue
        item_id = _resolve_item_id(performer.get("item_id"), available_item_ids)
        if item_id is not None and item_id not in resolved:
            resolved.append(item_id)
        if len(resolved) >= max_items:
            break
    return resolved[:max_items]


def forecast_data_to_frame(forecast_data: list[dict[str, Any]]) -> pd.DataFrame:
    """Parse ``forecast_data`` rows from ``back_testing.json``."""
    if not forecast_data:
        return pd.DataFrame(columns=["timestamp", "actual", "predicted", "lower_bound", "upper_bound"])
    frame = pd.DataFrame(forecast_data)
    frame["timestamp"] = pd.to_datetime(frame["timestamp"])
    return frame.sort_values("timestamp")


def per_window_metrics_table(
    per_window_metrics: list[dict[str, Any]],
    eval_metric: str,
) -> pd.DataFrame:
    """Build a per-cutoff metrics table aligned with AutoGluon backtest tutorials."""
    rows: list[dict[str, Any]] = []
    for index, window in enumerate(per_window_metrics):
        metrics = window.get("metrics") or {}
        metric_value = pick_window_metric(metrics, eval_metric)
        rows.append(
            {
                "cutoff": window.get("cutoff"),
                "window_id": window.get("window_id", index),
                "test_start": window.get("test_start", ""),
                "test_end": window.get("test_end", ""),
                eval_metric: metric_value,
            }
        )
    return pd.DataFrame(rows)


def mean_window_metric(per_window_metrics: list[dict[str, Any]], eval_metric: str) -> float | None:
    """Return the mean ``eval_metric`` across validation windows (each window is all-series)."""
    table = per_window_metrics_table(per_window_metrics, eval_metric)
    if eval_metric not in table.columns:
        return None
    values = table[eval_metric].dropna()
    if values.empty:
        return None
    mean = float(values.mean())
    return mean if math.isfinite(mean) else None


def _present_frame(frame: pd.DataFrame) -> None:
    try:
        from IPython.display import display

        display(frame)
    except ImportError:
        print(frame.to_string(index=False))


def _show_overall_backtest_summary(
    per_window_metrics: list[dict[str, Any]],
    eval_metric: str,
    *,
    num_series_evaluated: int | None = None,
) -> None:
    """Print mean ``eval_metric`` across windows (AutoGluon-style cross-series aggregate)."""
    overall = mean_window_metric(per_window_metrics, eval_metric)
    if overall is None:
        return
    window_count = len(per_window_metrics)
    window_label = "window" if window_count == 1 else "windows"
    summary = f"Overall {eval_metric} (mean across {window_count} validation {window_label}, all series): {overall:.6g}"
    if num_series_evaluated is not None:
        summary += f" | Series evaluated: {num_series_evaluated}"
    print(summary)


def _show_per_window_metrics(
    per_window_metrics: list[dict[str, Any]],
    eval_metric: str,
    *,
    num_series_evaluated: int | None = None,
) -> None:
    """Print cutoff scores and show a table (AutoGluon uses print + pivot, not bar charts)."""
    table = per_window_metrics_table(per_window_metrics, eval_metric)
    if eval_metric not in table.columns or table[eval_metric].notna().sum() == 0:
        print(f"No finite {eval_metric} values per validation window.")
        return

    for _, row in table.iterrows():
        value = row[eval_metric]
        if pd.notna(value):
            print(f"Cutoff {row['cutoff']}: {eval_metric} = {value}")

    _present_frame(table)
    _show_overall_backtest_summary(
        per_window_metrics,
        eval_metric,
        num_series_evaluated=num_series_evaluated,
    )


def _style_date_axis(ax: Any) -> None:
    """Rotate x labels to reduce overlap; keep matplotlib's default date formatting."""
    plt, _ = _matplotlib()
    ax.tick_params(axis="x", labelsize=8, rotation=45)
    plt.setp(ax.get_xticklabels(), ha="right")


def _resolve_cutoff_timestamp(cutoff_start: str | None, timestamps: pd.Series) -> pd.Timestamp | None:
    """Map a window ``test_start`` bound onto plotted timestamps.

    ``test_start`` is stored as YYYY-MM-DD for compact tables; for hourly (or sub-daily)
    data that resolves to midnight and can sit left of the first plotted point.
    """
    if timestamps.empty:
        return pd.to_datetime(cutoff_start, utc=True) if cutoff_start else None

    first_ts = pd.Timestamp(timestamps.min())
    if first_ts.tzinfo is not None:
        first_ts = first_ts.tz_convert("UTC").tz_localize(None)

    if not cutoff_start:
        return first_ts

    cutoff = pd.to_datetime(cutoff_start, utc=True)
    if cutoff.tzinfo is not None:
        cutoff = cutoff.tz_convert("UTC").tz_localize(None)

    if cutoff.normalize() == cutoff and cutoff < first_ts:
        return first_ts
    return cutoff


def _target_column_name(data: pd.DataFrame) -> str:
    if data.columns.empty:
        raise ValueError("Cannot extract target column name from DataFrame with no columns")
    return str(data.columns[0])


def _point_forecast_column(predictions: pd.DataFrame) -> str:
    if "0.5" in predictions.columns:
        return "0.5"
    if "mean" in predictions.columns:
        return "mean"
    raise ValueError("predictions must include a 'mean' or '0.5' forecast column")


def plot_timeseries_forecasts(
    data: pd.DataFrame,
    predictions: pd.DataFrame,
    *,
    item_ids: list[Any] | None = None,
    quantile_levels: list[float] | None = None,
    max_history_length: int | None = None,
    target: str | None = None,
) -> None:
    """Plot history and forecast quantiles (AutoGluon ``predictor.plot``-compatible axes)."""
    plt, _ = _matplotlib()
    quantile_levels = quantile_levels or [0.1, 0.9]
    q_low, q_high = (str(level) for level in quantile_levels[:2])
    point_column = _point_forecast_column(predictions)

    if item_ids is None:
        if hasattr(data, "item_ids"):
            plot_ids = list(data.item_ids)[:4]
        elif isinstance(data.index, pd.MultiIndex):
            plot_ids = list(data.index.get_level_values(0).unique()[:4])
        else:
            plot_ids = [None]
    else:
        plot_ids = list(item_ids)

    target_name = target or _target_column_name(data)
    count = max(len(plot_ids), 1)
    figure, axes = plt.subplots(1, count, figsize=(max(8.0, 4.0 * count), 4.5), squeeze=False)

    for index, item_id in enumerate(plot_ids):
        axis = axes[0, index]
        if item_id is None:
            history = data
            preds = predictions
        else:
            history = data.loc[item_id]
            preds = predictions.loc[item_id]

        if max_history_length:
            history = history.iloc[-max_history_length:]

        axis.plot(history.index, history.iloc[:, 0], label="Observed", color=_NEUTRAL)
        point_forecast = preds[point_column]
        axis.plot(
            preds.index,
            point_forecast,
            marker="s",
            linestyle="--",
            color=_ACCENT,
            label="Forecast",
        )

        if q_low in preds.columns and q_high in preds.columns:
            axis.fill_between(
                preds.index,
                preds[q_low],
                preds[q_high],
                alpha=0.1,
                color=_ACCENT,
                label=f"P{float(q_low) * 100:.0f}-P{float(q_high) * 100:.0f} interval",
            )

        if not preds.index.empty:
            axis.axvline(
                preds.index[0],
                color=_CUTOFF,
                linestyle=":",
                alpha=0.8,
                label="Forecast start",
            )

        title = str(item_id) if item_id is not None else target_name
        axis.set(title=title, xlabel="Date", ylabel=target_name if index == 0 else "")
        axis.legend(loc="best", fontsize=8)
        axis.grid(linestyle="--", alpha=0.3)
        _style_date_axis(axis)

    figure.tight_layout(rect=[0, 0.08, 1, 0.96])
    plt.show()


def _interval_label(frame: pd.DataFrame) -> str:
    if {"lower_quantile", "upper_quantile"}.issubset(frame.columns):
        lower = frame["lower_quantile"].dropna()
        upper = frame["upper_quantile"].dropna()
        if not lower.empty and not upper.empty:
            lo = float(lower.iloc[0])
            hi = float(upper.iloc[0])
            return f"{lo * 100:.0f}%-{hi * 100:.0f}% interval"
    return "10%-90% interval"


def _draw_cutoff(ax: Any, cutoff_start: str | None, timestamps: pd.Series | None = None) -> None:
    if timestamps is not None:
        cutoff = _resolve_cutoff_timestamp(cutoff_start, timestamps)
    elif cutoff_start:
        cutoff = pd.to_datetime(cutoff_start)
    else:
        return
    if cutoff is None:
        return
    ax.axvline(cutoff, color=_CUTOFF, linestyle="--", alpha=0.7, label="Cutoff")


def _draw_forecast(
    ax: Any,
    rows: list[dict[str, Any]],
    *,
    title: str,
    target: str,
    cutoff_start: str | None = None,
) -> None:
    frame = forecast_data_to_frame(rows)
    if frame.empty:
        raise ValueError("forecast_data is empty")

    timestamps = frame["timestamp"]
    if "actual" in frame.columns and frame["actual"].notna().any():
        ax.plot(timestamps, frame["actual"], marker="o", color=_NEUTRAL, label=f"Actual ({target})")
    ax.plot(timestamps, frame["predicted"], marker="s", linestyle="--", color=_ACCENT, label="Predicted")

    if {"lower_bound", "upper_bound"}.issubset(frame.columns):
        lower, upper = frame["lower_bound"], frame["upper_bound"]
        if lower.notna().any() and upper.notna().any():
            ax.fill_between(timestamps, lower, upper, alpha=0.1, color=_ACCENT, label=_interval_label(frame))

    _draw_cutoff(ax, cutoff_start, timestamps)

    ax.set(title=title, xlabel="Date", ylabel=target)
    ax.legend(loc="best", fontsize=8)
    ax.grid(linestyle="--", alpha=0.3)
    _style_date_axis(ax)


def render_back_testing_metrics(back_testing: dict[str, Any]) -> None:
    """Print per-window backtest scores and summary table (no plots)."""
    eval_metric = back_testing.get("eval_metric", "mean_absolute_scaled_error")
    per_window = back_testing.get("per_window_metrics") or []
    series_analysis = back_testing.get("series_analysis") or {}
    num_series = series_analysis.get("num_series_evaluated")
    _show_per_window_metrics(
        per_window,
        eval_metric,
        num_series_evaluated=num_series if isinstance(num_series, int) else None,
    )


def render_back_testing_forecast_charts(back_testing: dict[str, Any]) -> None:
    """Render best/worst holdout forecast matplotlib panels."""
    eval_metric = back_testing.get("eval_metric", "mean_absolute_scaled_error")
    target = back_testing.get("target", "target")
    per_window = back_testing.get("per_window_metrics") or []
    series_analysis = back_testing.get("series_analysis") or {}

    plotted_ids: set[Any] = set()
    plt, _ = _matplotlib()
    for heading, role in (("Best", "best_performer"), ("Worst", "worst_performer")):
        performer = series_analysis.get(role)
        if not performer:
            continue
        item_id = performer.get("item_id")
        if item_id in plotted_ids:
            continue
        plotted_ids.add(item_id)

        item_windows = performer.get("windows") or []
        if not item_windows:
            continue

        count = len(item_windows)
        figure, axes = plt.subplots(1, count, figsize=(max(8.0, 4.0 * count), 4.2), squeeze=False)
        window_starts = {
            window.get("window_id"): window.get("test_start")
            for window in per_window
            if window.get("window_id") is not None
        }
        for index, window in enumerate(item_windows):
            axis = axes[0, index]
            window_id = window.get("window_id", index)
            metric_value = pick_window_metric(window.get("metrics") or {}, eval_metric)
            metric_label = _metric_display_name(eval_metric)
            metric_text = f"{metric_label}={metric_value:.3g}" if metric_value is not None else metric_label
            try:
                _draw_forecast(
                    axis,
                    window.get("forecast_data") or [],
                    title=f"Window {window_id} ({metric_text})",
                    target=target,
                    cutoff_start=window_starts.get(window_id),
                )
            except ValueError as exc:
                print(exc)
            if index:
                axis.set_ylabel("")

        figure.suptitle(f"{heading} performer: {item_id}", fontsize=12)
        figure.tight_layout(rect=[0, 0.04, 1, 0.92])
        plt.show()


def render_back_testing_charts(back_testing: dict[str, Any]) -> None:
    """Render per-window metrics and best/worst holdout forecast panels."""
    render_back_testing_metrics(back_testing)
    render_back_testing_forecast_charts(back_testing)




import json
from pathlib import Path

back_testing_path = Path(best_model_path) / "metrics" / "back_testing.json"
if not back_testing_path.is_file():
    print("back_testing.json was not found for this model (older pipeline runs may omit it).")
else:
    with back_testing_path.open(encoding="utf-8") as f:
        back_testing = json.load(f)
    render_back_testing_metrics(back_testing)

<a id="back-testing-charts"></a>
### Back-testing charts

Best/worst series holdout forecasts from ``metrics/back_testing.json``. Run **Back-testing metrics** first (defines helpers). Uses **matplotlib** like tabular ROC/PR curves; no ``TimeSeriesPredictor`` required.

In [ ]:
back_testing_path = Path(best_model_path) / "metrics" / "back_testing.json"
if not back_testing_path.is_file():
    print("Skipping backtest charts: back_testing.json not found.")
elif "render_back_testing_forecast_charts" not in globals():
    print("Run the Back-testing metrics cell first.")
elif "back_testing" not in globals():
    print("Run the Back-testing metrics cell first (defines back_testing variable).")
else:
    render_back_testing_forecast_charts(back_testing)

<a id="load-the-predictor"></a>
## Load the predictor

Load a **`TimeSeriesPredictor`** from the downloaded `predictor/` directory (not `TabularPredictor`).

In [20]:
from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor

predictor = TimeSeriesPredictor.load(os.path.join(best_model_path, "predictor"), require_version_match=False)

<a id="refit-training-leaderboard"></a>
### Refit training leaderboard

Run after **Load the predictor**. Shows models trained in this refit step (often one row).

In [ ]:
# Models in this refit predictor (refit often trains one model)
predictor.leaderboard()


In [ ]:
predictor.info()

<a id="predict-the-values"></a>
## Forecast (`predict`)

Use the sample data below (embedded when this notebook was generated) as historical context for ``predict()``. When the model uses known covariates, values for the forecast horizon are rebuilt from ``predictor.make_future_data_frame()`` after parsing sample timestamps.

In [ ]:
"""Helpers for AutoGluon time series inference notebooks."""

from __future__ import annotations

from pathlib import Path
from typing import Any

import pandas as pd


def _coerce_sample_timestamps(score_df: pd.DataFrame, timestamp_column: str) -> pd.DataFrame:
    """Parse sample-row timestamps written as epoch milliseconds by ``DataFrame.to_json``."""
    series = score_df[timestamp_column]
    if not pd.api.types.is_numeric_dtype(series):
        out = score_df.copy()
        out[timestamp_column] = pd.to_datetime(series, errors="coerce")
        return out

    numeric = pd.to_numeric(series, errors="coerce")
    if numeric.isna().any():
        out = score_df.copy()
        out[timestamp_column] = pd.to_datetime(series, errors="coerce")
        return out

    if numeric.abs().max() > 1e11:
        out = score_df.copy()
        out[timestamp_column] = pd.to_datetime(numeric, unit="ms")
        return out

    out = score_df.copy()
    out[timestamp_column] = pd.to_datetime(series, errors="coerce")
    return out


def _is_datetime_series(series: pd.Series) -> bool:
    """Return whether a series holds datetime values (works with test pandas mocks)."""
    dtype = getattr(series, "dtype", None)
    if dtype is not None and str(dtype).startswith("datetime64"):
        return True
    api = getattr(pd, "api", None)
    if api is not None:
        return api.types.is_datetime64_any_dtype(series)
    return False


def _json_records(df: pd.DataFrame) -> list[dict[str, Any]]:
    """Convert a DataFrame to JSON-safe row dicts (ISO timestamps)."""
    out = df.copy()
    for col in out.columns:
        if _is_datetime_series(out[col]):
            out[col] = out[col].dt.strftime("%Y-%m-%dT%H:%M:%S.%f").str.rstrip("0").str.rstrip(".")
    return out.to_dict(orient="records")


def _future_horizon_frame(future_cov: pd.DataFrame) -> pd.DataFrame:
    """Normalize ``make_future_data_frame`` output to flat item_id + timestamp columns."""
    if {"item_id", "timestamp"}.issubset(future_cov.columns):
        out = future_cov.copy()
        out["timestamp"] = pd.to_datetime(out["timestamp"])
        return out

    if isinstance(future_cov.index, pd.MultiIndex):
        out = future_cov.reset_index()
        id_name = future_cov.index.names[0] or "item_id"
        ts_name = future_cov.index.names[1] or "timestamp"
        rename = {}
        if out.columns[0] != id_name:
            rename[out.columns[0]] = id_name
        if len(out.columns) > 1 and out.columns[1] != ts_name:
            rename[out.columns[1]] = ts_name
        if rename:
            out = out.rename(columns=rename)
        out = out.rename(columns={id_name: "item_id", ts_name: "timestamp"})
        out["timestamp"] = pd.to_datetime(out["timestamp"])
        return out

    raise ValueError(
        "future_cov must come from make_future_data_frame (item_id and timestamp columns) "
        "or use a MultiIndex (item_id, timestamp)."
    )


def _historical_covariates_long(
    ts_df: pd.DataFrame,
    known_covariates_names: list[str],
) -> pd.DataFrame:
    """Expand ts_df covariates to flat item_id + timestamp columns."""
    if not isinstance(ts_df.index, pd.MultiIndex):
        raise ValueError("ts_df must use a MultiIndex (item_id, timestamp).")

    hist = ts_df[known_covariates_names].reset_index()
    id_name = ts_df.index.names[0] or "item_id"
    ts_name = ts_df.index.names[1] or "timestamp"
    hist = hist.rename(columns={hist.columns[0]: id_name, hist.columns[1]: ts_name})
    hist = hist.rename(columns={id_name: "item_id", ts_name: "timestamp"})
    hist["timestamp"] = pd.to_datetime(hist["timestamp"])
    return hist


def fill_known_covariates_on_future_frame(
    future_cov: pd.DataFrame,
    ts_df: pd.DataFrame,
    known_covariates_names: list[str],
) -> pd.DataFrame:
    """Fill known covariates on the flat forecast frame returned by ``make_future_data_frame``."""
    missing = [col for col in known_covariates_names if col not in ts_df.columns]
    if missing:
        raise ValueError(f"Known covariates missing from ts_df: {missing}")

    future_df = _future_horizon_frame(future_cov)[["item_id", "timestamp"]].copy()
    hist = _historical_covariates_long(ts_df, known_covariates_names)
    last_by_item = hist.groupby("item_id", sort=False)[known_covariates_names].last()

    for col in known_covariates_names:
        merged = future_df.merge(
            hist[["item_id", "timestamp", col]],
            on=["item_id", "timestamp"],
            how="left",
        )
        future_df[col] = merged[col].fillna(future_df["item_id"].map(last_by_item[col]))

    return future_df


import pandas as pd

sample = {'id_column': 'item_id', 'timestamp_column': 'timestamp', 'known_covariates_names': [], 'history': [{'item_id': 'H99', 'timestamp': '1750-01-29T23:00:00', 'target': 27926.0}, {'item_id': 'H99', 'timestamp': '1750-01-30T00:00:00', 'target': 26744.0}, {'item_id': 'H99', 'timestamp': '1750-01-30T01:00:00', 'target': 25829.0}, {'item_id': 'H99', 'timestamp': '1750-01-30T02:00:00', 'target': 25421.0}, {'item_id': 'H99', 'timestamp': '1750-01-30T03:00:00', 'target': 23252.0}], 'known_covariates': None}

history = pd.DataFrame(sample["history"])
history = _coerce_sample_timestamps(history, sample["timestamp_column"])

ts_df = TimeSeriesDataFrame.from_data_frame(
    history,
    id_column=sample["id_column"],
    timestamp_column=sample["timestamp_column"],
)

known_covariates = None
covariate_names = sample.get("known_covariates_names") or []
if covariate_names and all(col in history.columns for col in covariate_names):
    future_cov = predictor.make_future_data_frame(ts_df)
    known_covariates = fill_known_covariates_on_future_frame(future_cov, ts_df, covariate_names)

ts_df.head()


Run **`predict`** to get quantile forecasts. 

In [ ]:
forecasts = predictor.predict(
    ts_df,
    known_covariates=known_covariates,
    use_cache=False,
)
forecasts


<a id="plot-forecasted-data"></a>
### Plot forecasted data

Visualize historical values and the forecast distribution (mean and 10%/90% interval). Uses **matplotlib** with the same date-axis styling as back-testing charts (aligned with [AutoGluon ``predictor.plot()``](https://auto.gluon.ai/stable/tutorials/timeseries/forecasting-quick-start.html)). When ``back_testing.json`` is available (run **Back-testing metrics** first), plots the same best/worst series as the backtest charts; otherwise uses the first series in the embedded sample.

In [ ]:
import json
from pathlib import Path

available_item_ids = list(ts_df.item_ids)
plot_item_ids = available_item_ids[:4]

back_testing_for_plot = globals().get("back_testing")
if back_testing_for_plot is None:
    back_testing_path = Path(best_model_path) / "metrics" / "back_testing.json"
    if back_testing_path.is_file():
        with back_testing_path.open(encoding="utf-8") as f:
            back_testing_for_plot = json.load(f)

if back_testing_for_plot and "backtest_highlight_item_ids" in globals():
    highlight_ids = backtest_highlight_item_ids(back_testing_for_plot, available_item_ids)
    if highlight_ids:
        plot_item_ids = highlight_ids
        print("Plotting backtest best/worst performers:", plot_item_ids)
    else:
        print("Backtest performers not in sample; plotting first sample series.")
elif back_testing_for_plot:
    print("Run Back-testing metrics first to align plot series with backtest charts.")

if isinstance(ts_df.index, pd.MultiIndex):
    history_limit = min(200, int(ts_df.groupby(level=0).size().min()))
else:
    history_limit = min(200, len(ts_df))

if "plot_timeseries_forecasts" not in globals():
    print("Run Back-testing metrics first (defines plot helpers).")
else:
    plot_timeseries_forecasts(
        data=ts_df,
        predictions=forecasts,
        item_ids=plot_item_ids,
        quantile_levels=[0.1, 0.9],
        max_history_length=history_limit,
    )

<a id="summary-and-next-steps"></a>
## Summary and next steps

**Summary:** Loaded an AutoGluon Time Series predictor from S3 and ran **`predict()`** to forecast `prediction_length` steps ahead per item.

**Next steps:**
- Run predictions on your own data (ensure columns match the training schema).
- Optionally create the Predictor's online deployment using KServe custom runtime.

---